In [1]:
import pandas as pd

df = pd.read_csv("incident_event_log.csv")

In [2]:
df.head()
df.shape
df.info()
df.describe()
df.columns

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 141712 entries, 0 to 141711
Data columns (total 36 columns):
 #   Column                   Non-Null Count   Dtype 
---  ------                   --------------   ----- 
 0   number                   141712 non-null  object
 1   incident_state           141712 non-null  object
 2   active                   141712 non-null  bool  
 3   reassignment_count       141712 non-null  int64 
 4   reopen_count             141712 non-null  int64 
 5   sys_mod_count            141712 non-null  int64 
 6   made_sla                 141712 non-null  bool  
 7   caller_id                141712 non-null  object
 8   opened_by                141712 non-null  object
 9   opened_at                141712 non-null  object
 10  sys_created_by           141712 non-null  object
 11  sys_created_at           141712 non-null  object
 12  sys_updated_by           141712 non-null  object
 13  sys_updated_at           141712 non-null  object
 14  contact_type        

Index(['number', 'incident_state', 'active', 'reassignment_count',
       'reopen_count', 'sys_mod_count', 'made_sla', 'caller_id', 'opened_by',
       'opened_at', 'sys_created_by', 'sys_created_at', 'sys_updated_by',
       'sys_updated_at', 'contact_type', 'location', 'category', 'subcategory',
       'u_symptom', 'cmdb_ci', 'impact', 'urgency', 'priority',
       'assignment_group', 'assigned_to', 'knowledge',
       'u_priority_confirmation', 'notify', 'problem_id', 'rfc', 'vendor',
       'caused_by', 'closed_code', 'resolved_by', 'resolved_at', 'closed_at'],
      dtype='object')

In [3]:
df.isnull().sum()
df.duplicated().sum()
df["number"].nunique()

24918

In [4]:
date_columns = [
    "opened_at",
    "sys_created_at",
    "sys_updated_at",
    "resolved_at",
    "closed_at"
]

for col in date_columns:
    df[col] = pd.to_datetime(
        df[col],
        format="%d/%m/%Y %H:%M",
        errors="coerce"
    )

In [5]:
numeric_columns = [
    "reassignment_count",
    "reopen_count",
    "sys_mod_count"
]

for col in numeric_columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

In [6]:
categorical_columns = [
    "category",
    "subcategory",
    "assignment_group",
    "assigned_to",
    "location"
]

for col in categorical_columns:
    df[col] = df[col].fillna("Unknown")

In [7]:
df["resolution_hours"] = (
    df["resolved_at"] - df["opened_at"]
).dt.total_seconds() / 3600

In [8]:
df["closure_hours"] = (
    df["closed_at"] - df["opened_at"]
).dt.total_seconds() / 3600

In [9]:
df["resolution_days"] = df["resolution_hours"] / 24

In [10]:
(df["resolution_hours"] < 0).sum()
(df["closure_hours"] < 0).sum()
df["number"].nunique()

24918

In [11]:
df.groupby("priority")["number"].nunique()

,number
priority,
1 - Critical,352
2 - High,458
3 - Moderate,23738
4 - Low,785


In [12]:
df.groupby("category")["number"].nunique()

,number
category,
?,39
Category 10,3
Category 12,1
Category 13,129
Category 14,1
Category 15,1
Category 16,2
Category 17,89
Category 19,255


In [17]:
df["sla_status"] = df["made_sla"].map({
    True: "Met SLA",
    False: "Missed SLA"
})

In [18]:
df["sla_status"] = df["made_sla"].map({
    1: "Met SLA",
    0: "Missed SLA"
})

In [19]:
df["sla_status"].value_counts()

,count
sla_status,


In [20]:
df.groupby("sla_status")["number"].nunique()

,number
sla_status,


In [14]:
df.groupby("assignment_group")["number"].nunique()

,number
assignment_group,
?,3855
Group 10,525
Group 11,1
Group 12,206
Group 13,46
...,...
Group 8,2
Group 80,9
Group 81,53


In [15]:
df.groupby("priority")["resolution_hours"].mean()

,resolution_hours
priority,
1 - Critical,243.275908
2 - High,175.665657
3 - Moderate,268.133298
4 - Low,407.471736


In [16]:
incident_level = (
    df.groupby("number")["reopen_count"]
      .max()
      .reset_index()
)

reopened_incidents = incident_level[
    incident_level["reopen_count"] > 0
]

len(reopened_incidents)

275